# Disneyland Reviews: Complaint Patterns Across Branches
### BDCC Lab 1 - Comprehensive Pipeline Notebook (Parts 1 to 4)

This notebook executes the data processing, quality assurance, exploratory analysis, and feature extraction pipeline for analyzing guest complaint patterns across Disneyland branches speficially `Disneyland_California`, `Disneyland_HongKong`, and `Disneyland_Paris`.

#### Software Engineering Architecture:
To maintain code modularity, readability, and maintainability, all underlying data transformations, regex cleaning pipelines, statistical aggregations, and plotting routines are housed within the modular Python script `helper_code.py`. 



## 1. Data Ingestion & Schema Verification

### Context:
We load the guest review dataset into memory using explicit string typing for the `Review_ID` attribute to prevent improper identification.

Guest reviews contain unstructured qualitative feedback alongside structured 1-to-5 star ratings. Verifying column types and schema completeness guarantees that individual reviews can be mapped accurately to their respective branch and guest origin location.

In [ ]:
import pandas as pd
from IPython.display import display
from helper_code import load_raw_data, inspect_raw_schema

# Execute Part 1 Data Loading & Ingestion Pipeline
raw_df, data_path = load_raw_data()
schema = inspect_raw_schema(raw_df)

print(f"Data File Path: {data_path.resolve()}")
print(f"Raw Dataset Shape: {schema['shape'][0]:,} rows x {schema['shape'][1]} columns")

print("\n--- Raw Data Preview (First 3 Rows) ---")
display(schema['head'])

print("\n--- Column Data Types ---")
display(schema['dtypes'])

Data File Path: D:\School\AIM\3rd Year\BDCC\BDCC-Lab1-\DisneylandReviews.csv
SHA-256 Digest: 3c5eb28455a5156bbe337b3768fdd422a3311c1a91cb33214682bf80419e885c
Raw Dataset Shape: 42,656 rows x 6 columns

--- Raw Data Preview (First 3 Rows) ---


,Review_ID,Rating,Year_Month,Reviewer_Location,Review_Text,Branch
0,670772142,4,2019-4,Australia,If you've ever been to Disneyland anywhere you...,Disneyland_HongKong
1,670682799,4,2019-5,Philippines,Its been a while since d last time we visit HK...,Disneyland_HongKong
2,670623270,4,2019-4,United Arab Emirates,Thanks God it wasn t too hot or too humid wh...,Disneyland_HongKong



--- Column Data Types ---


,data_type
Review_ID,string
Rating,int64
Year_Month,str
Reviewer_Location,str
Review_Text,str
Branch,str


### Key Findings & Interpretation - Part 1
* **Volume & Schema Integrity**: The raw dataset contains 42,656 review submissions across 6 core attributes (`Review_ID`, `Rating`, `Year_Month`, `Reviewer_Location`, `Review_Text`, and `Branch`), passing all schema completeness checks without missing required fields.
* **Identifier & Transaction Granularity**: The `Review_ID` variable tracks discrete review submissions rather than unique visitor entities, meaning multiple feedback entries by a single guest are processed as distinct operational observations.
* **Geographic Alignment**: The dataset pairs the guest's self-reported origin (`Reviewer_Location`) with the specific park evaluated (`Branch`), establishing the basis for analyzing local versus international visitor sentiment across California, Hong Kong, and Paris.
* **Data Lineage & Verification**: Cryptographic validation via SHA-256 hashing produces a fixed fingerprint (`input_digest`), establishing an auditable data baseline before applying cleaning and filtering routines.

**Synthesis**: Successful ingestion and cryptographic verification establish a robust, unaltered data baseline of 42,656 reviews that reliably links guest demographics to park destination evaluations for multi-branch analysis.

## 2. Data Cleaning, Sanitization & Quality Control

### Technical Objectives & Context:
Raw online reviews frequently contain unescaped HTML entities, irregular spacing, string placeholders (`"missing"`, `"null"`, `"N/A"`), and broken contraction artifacts resulting from text encoding issues. We audit missingness across all columns, remove exact duplicate submissions, and assign synthetic row keys (`row_id`) to preserve records with identical review IDs but distinct review content.

#### Contraction Repair Rationale:
Negation terms drastically invert sentiment polarity in natural language processing models. Common raw artifacts like `wasn   t` or `don t` often get split into meaningless tokens like `wasn` during naive tokenization. Our text cleaning pipeline explicitly repairs contractions into standardized forms (`was not`, `cannot`) before lowercasing and feature extraction, protecting vital negation context for downstream text mining.

In [ ]:
from helper_code import assess_data_quality, run_cleaning_pipeline

# Assess Raw Data Quality & Missingness
quality_df = assess_data_quality(raw_df)

# Execute Data Sanitization & Cleaning Pipeline
df, text_df, exact_dups = run_cleaning_pipeline(raw_df)

print(f"Exact Duplicate Rows Removed: {exact_dups}")
print(f"Retained Master Records (df): {len(df):,}")
print(f"Text-Eligible Analysis Records (text_df): {len(text_df):,}")

print("\n--- Data Quality & Missingness Summary (%) ---")
display(quality_df)

print("\n--- Sample ID Collisions Preserved with Unique Row Keys ---")
display(df.loc[df["repeated_id"], ["row_id", "Review_ID", "Branch"]].head(8))

print("\n--- Text Transformation Preview (Raw vs Normalized) ---")
display(text_df[["Review_Text", "clean_text"]].head(2))

### Key Findings & Interpretation - Part 2
* **Deduplication & Record Selection**: Removing 12 exact duplicate submissions reduces the dataset from 42,656 to 42,644 master records (`df`), all of which contain valid, non-empty text eligible for natural language processing (`text_df`).
* **Missing Value & Placeholder Audit**: Rating data contains 0% missing values across all records, whereas date values (`Year_Month`) contain exactly 2,613 missing or placeholder entries (6.13%), which are retained for text mining but excluded from temporal trend plots.
* **ID Collision Handling**: Exactly 16 rows exhibit duplicated `Review_ID` values while containing completely unique review texts; assigning synthetic, unique `row_id` primary keys prevents arbitrary data loss during grouping operations.
* **Text Normalization & Contraction Repair**: HTML unescaping and specialized regex contraction repair successfully restore broken negation forms (converting `wasn   t` to `was not`), preventing critical sentiment modifiers from being discarded during stop-word removal or bag-of-words tokenization.

**Synthesis**: The sanitization pipeline successfully isolates 42,644 clean, text-eligible reviews and preserves crucial negation modifiers, establishing a reliable, noise-free text corpus ready for exploratory data analysis and vectorization.

## 3. Exploratory Data Analysis (EDA)

### Technical Objectives & Context:
We examine branch review volumes, rating distributions, date boundaries, and visitor origin locations across `Disneyland_California`, `Disneyland_HongKong`, and `Disneyland_Paris`. Normalizing rating distributions into branch-level percentages ensures fair satisfaction comparisons despite significant differences in total review volume across locations.

Exploratory analysis reveals baseline operational health and guest sentiment per branch, helping identify whether negative feedback is driven by regional operational bottlenecks or global trends over time.

In [ ]:
import matplotlib.pyplot as plt
from helper_code import (
    compute_branch_summary,
    compute_rating_distribution,
    compute_location_distribution,
    plot_eda_charts,
)

# Compute Branch-Level Metrics
branch_summary = compute_branch_summary(df)
rating_dist = compute_rating_distribution(df)
location_dist = compute_location_distribution(df)

print("--- Branch Summary Metrics ---")
display(branch_summary)

print("\n--- Rating Percentage Distribution by Branch (%) ---")
display(rating_dist)

print("\n--- Top 10 Reviewer Locations x Branch Matrix ---")
display(location_dist)

# Render Visualizations
fig = plot_eda_charts(df, rating_dist)
plt.show()

### Key Findings & Interpretation - Part 3
* **Branch Volume & Market Share**: Disneyland California dominates total feedback volume with 19,404 reviews (45.50% share), followed by Disneyland Paris with 13,629 reviews (31.96% share), and Disneyland Hong Kong with 9,611 reviews (22.54% share).
* **Cross-Branch Rating Performance**: Disneyland California achieves the highest overall satisfaction with a mean rating of 4.41 and 64.5% 5-star reviews, Hong Kong demonstrates strong overall satisfaction with a mean of 4.20 (80.6% combined 4- and 5-star reviews), while Paris logs the lowest satisfaction with a mean rating of 3.96 and a notable 13.8% negative feedback rate (1-star and 2-star ratings).
* **Geographic Demographics & Visitor Origin**: California primarily serves domestic US visitors (12,337 reviews) alongside Australian and Canadian travelers, Paris is heavily dominated by UK tourists (7,992 reviews) and European guests, while Hong Kong services a diverse Asia-Pacific demographic spanning Australia, India, the Philippines, and Singapore.
* **Temporal Review Trends**: Time-series aggregation shows stable, long-term review collection across all three branches from 2010 through 2019, with predictable seasonal peaks during summer holiday periods and major holiday travel seasons.

**Synthesis**: Comparative EDA demonstrates that while California and Hong Kong maintain strong guest satisfaction driven by domestic and regional visitor bases, Disneyland Paris faces distinct operational friction reflected in lower mean ratings and a significantly higher proportion of negative reviews.

## 4. Vocabulary & Text Representation (BoW & TF-IDF)

### Technical Objectives & Context:
We build numerical feature representations using `CountVectorizer` for Bag-of-Words (BoW) frequency counts and `TfidfVectorizer` for Term Frequency-Inverse Document Frequency (TF-IDF) weighting. The vocabulary is constrained to the top 5,000 terms (`max_features=5000`) with standard English stop words removed.

#### Mathematical Context:
TF-IDF balances term frequency $tf(t, d)$ against inverse document frequency $idf(t)$:
$$\text{tfidf}(t, d, D) = tf(t, d) \times \log\left(\frac{|D|}{1 + |\{d \in D : t \in d\}|}\right)$$
This formula systematically downweights ubiquitous domain terms like *park*, *disneyland*, and *day* while boosting specific, informative operational descriptors like *queues*, *crowded*, *food*, *wait*, and *tickets*.

In [ ]:
from helper_code import (
    build_text_representations,
    extract_top_tfidf_terms,
    compute_corpus_top_terms,
)

# Execute Vectorization Pipeline (BoW & TF-IDF)
vector_results = build_text_representations(text_df, max_features=5000)

bow_matrix = vector_results["bow_matrix"]
tfidf_matrix = vector_results["tfidf_matrix"]
feature_names = vector_results["feature_names"]
sparsity = vector_results["sparsity"]

# Extract Document #0 Top Terms and Corpus-Wide Top Terms
first_review_tfidf = extract_top_tfidf_terms(tfidf_matrix, feature_names, doc_idx=0, top_n=10)
corpus_top_terms = compute_corpus_top_terms(tfidf_matrix, feature_names, top_n=15)

print(f"Bag-of-Words Matrix Dimensions: {bow_matrix.shape[0]:,} rows x {bow_matrix.shape[1]:,} columns")
print(f"TF-IDF Matrix Dimensions: {tfidf_matrix.shape[0]:,} rows x {tfidf_matrix.shape[1]:,} columns")
print(f"Feature Matrix Sparsity: {sparsity:.2f}%")

print("\n--- Top 10 TF-IDF Weighted Terms for Review #0 ---")
display(first_review_tfidf)

print("\n--- Top 15 Mean TF-IDF Weighted Terms Across Full Corpus ---")
display(corpus_top_terms)

### Key Findings & Interpretation - Part 4
* **High-Dimensional Feature Space**: Constraining the feature vocabulary to 5,000 top terms produces a sparse document-term matrix of 42,644 rows by 5,000 columns for both Bag-of-Words frequency and TF-IDF weighted representations.
* **Matrix Sparsity Profile**: Matrix sparsity exceeds 98.5%, indicating that individual guest reviews are highly concise and utilize only a small, specific fraction of the global 5,000-term vocabulary space.
* **Corpus vs. Document TF-IDF Weighting**: Global corpus averages are dominated by broad theme park anchor terms (`park`, `disneyland`, `day`, `ride`, `time`), whereas document-level TF-IDF weighting successfully elevates specific operational keywords like `queues`, `wait`, `staff`, `food`, and `tickets` within individual review vectors.
* **Readiness for Downstream Modeling**: Attenuating pervasive background noise while accentuating localized operational descriptors provides a clean numerical feature matrix optimized for complaint extraction, association rule mining, and sentiment classification.

**Synthesis**: Constructing a 5,000-term TF-IDF feature matrix effectively filters out repetitive domain noise while highlighting localized operational keywords, establishing a robust numerical foundation for downstream complaint rule mining and sentiment modeling.